# 2b — Apply Manual Symptom Overrides

This notebook combines:

```text
conditions_symptom_parsed.json
        +
manual_symptom_overrides.json
        ↓
conditions_symptom_parsed_with_manual_overrides.json
```

It does **not** re-run symptom parsing.

By default, a manual override is applied **only when the automatic parser produced zero symptom phrases**. This preserves successful automatic parser results and uses the manual file only to fix known misses.


## 1. Imports and paths

In [37]:
from pathlib import Path
from collections import Counter
import json


def find_project_root() -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "app").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Expected an app/ folder."
    )


PROJECT_ROOT = find_project_root()

PARSED_PATH = (
    PROJECT_ROOT
    / "app"
    / "dataset"
    / "output"
    / "preprocessing"
    / "conditions_symptom_parsed.json"
)

OVERRIDES_PATH = (
    PROJECT_ROOT
    / "app"
    / "dataset"
    / "review"
    / "manual_symptom_overrides.json"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "app"
    / "dataset"
    / "output"
    / "preprocessing"
    / "conditions_symptom_parsed_with_manual_overrides.json"
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Parsed input:", PARSED_PATH)
print("Manual overrides:", OVERRIDES_PATH)
print("Output:", OUTPUT_PATH)


Project root: C:\Users\po\Downloads\medical-diagnosis
Parsed input: C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\preprocessing\conditions_symptom_parsed.json
Manual overrides: C:\Users\po\Downloads\medical-diagnosis\app\dataset\review\manual_symptom_overrides.json
Output: C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\preprocessing\conditions_symptom_parsed_with_manual_overrides.json


## 2. Load both input files

In [38]:
for path in [PARSED_PATH, OVERRIDES_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

with PARSED_PATH.open("r", encoding="utf-8") as f:
    parsed_conditions = json.load(f)

with OVERRIDES_PATH.open("r", encoding="utf-8") as f:
    raw_overrides = json.load(f)

if not isinstance(parsed_conditions, list):
    raise TypeError(
        "conditions_symptom_parsed.json must contain a list of condition records."
    )

print("Parsed conditions loaded:", len(parsed_conditions))
print("Manual override file type:", type(raw_overrides).__name__)


Parsed conditions loaded: 406
Manual override file type: dict


## 3. Normalize the manual-override file

The notebook supports either of these formats:

```json
{
  "Bone cancer": ["bone pain", "swelling"]
}
```

or:

```json
[
  {
    "condition": "Bone cancer",
    "symptom_phrases": ["bone pain", "swelling"]
  }
]
```


In [39]:
def normalize_overrides(raw):
    # Your manual override file has a wrapper:
    # {
    #     "_meta": {...},
    #     "overrides": {...}
    # }
    if isinstance(raw, dict) and "overrides" in raw:
        raw = raw["overrides"]

    overrides = {}

    # Format:
    # {
    #   "Bone cancer": ["bone pain", "swelling"]
    # }
    if isinstance(raw, dict):
        for condition, value in raw.items():

            if isinstance(value, list):
                phrases = value

            elif isinstance(value, dict):
                phrases = value.get("symptom_phrases", [])

            else:
                raise TypeError(
                    f"Unsupported override value for {condition!r}: "
                    f"{type(value).__name__}"
                )

            condition = str(condition).strip()

            phrases = [
                str(phrase).strip()
                for phrase in phrases
                if str(phrase).strip()
            ]

            if not condition:
                raise ValueError(
                    "Found a manual override with an empty condition name."
                )

            if not phrases:
                raise ValueError(
                    f"Manual override for {condition!r} has no symptom phrases."
                )

            overrides[condition] = phrases

    # Format:
    # [
    #   {
    #       "condition": "Bone cancer",
    #       "symptom_phrases": [...]
    #   }
    # ]
    elif isinstance(raw, list):
        for item in raw:

            if not isinstance(item, dict):
                raise TypeError(
                    "Each override item must be an object."
                )

            condition = str(
                item.get("condition", "")
            ).strip()

            phrases = item.get(
                "symptom_phrases",
                []
            )

            if not condition:
                raise ValueError(
                    "An override record is missing its condition name."
                )

            if not isinstance(phrases, list):
                raise TypeError(
                    f"symptom_phrases for {condition!r} must be a list."
                )

            phrases = [
                str(phrase).strip()
                for phrase in phrases
                if str(phrase).strip()
            ]

            if not phrases:
                raise ValueError(
                    f"Manual override for {condition!r} has no symptom phrases."
                )

            overrides[condition] = phrases

    else:
        raise TypeError(
            "Could not understand manual_symptom_overrides.json format."
        )

    return overrides


manual_overrides = normalize_overrides(raw_overrides)

print("Manual overrides loaded:", len(manual_overrides))

Manual overrides loaded: 60


## 4. Validate condition names before merging

Every manual override should match a condition in the parsed dataset. This protects against typos silently creating bad output.


In [40]:
parsed_names = [
    str(record.get("condition", "")).strip()
    for record in parsed_conditions
]

duplicate_conditions = [
    name
    for name, count in Counter(parsed_names).items()
    if name and count > 1
]

if duplicate_conditions:
    raise ValueError(
        "Duplicate condition names found in parsed dataset: "
        + ", ".join(duplicate_conditions[:10])
    )

parsed_name_set = set(parsed_names)

missing_override_conditions = sorted(
    set(manual_overrides) - parsed_name_set
)

if missing_override_conditions:
    raise ValueError(
        "These manual overrides do not match any parsed condition:\n"
        + "\n".join(missing_override_conditions)
    )

print("All manual override condition names match the parsed dataset ✅")


All manual override condition names match the parsed dataset ✅


## 5. Apply manual overrides

`APPLY_ONLY_WHEN_ZERO = True` matches the workflow used for this project:

- keep successful automatic parser results
- use manual overrides only for conditions where automatic parsing returned zero phrases


In [41]:
APPLY_ONLY_WHEN_ZERO = True

merged_conditions = []

applied = []
skipped_existing = []
unused_overrides = set(manual_overrides)

for record in parsed_conditions:
    updated = dict(record)
    condition = str(record["condition"]).strip()

    automatic_phrases = record.get("symptom_phrases", [])
    automatic_count = len(automatic_phrases)

    if condition in manual_overrides:
        override_phrases = manual_overrides[condition]

        should_apply = (
            automatic_count == 0
            if APPLY_ONLY_WHEN_ZERO
            else True
        )

        if should_apply:
            updated["symptom_phrases"] = override_phrases
            updated["symptom_phrase_count"] = len(override_phrases)
            updated["parser_method"] = "manual_override"
            updated["parser_needs_review"] = False

            applied.append(condition)
            unused_overrides.discard(condition)

        else:
            skipped_existing.append(condition)
            unused_overrides.discard(condition)

    merged_conditions.append(updated)

print("Overrides available:", len(manual_overrides))
print("Overrides applied:", len(applied))
print("Skipped because automatic parser already had phrases:", len(skipped_existing))
print("Unused overrides:", len(unused_overrides))


Overrides available: 60
Overrides applied: 60
Skipped because automatic parser already had phrases: 0
Unused overrides: 0


## 6. Validate the merged output

In [42]:
invalid_counts = []

for record in merged_conditions:
    phrases = record.get("symptom_phrases", [])
    stored_count = record.get("symptom_phrase_count", 0)

    if stored_count != len(phrases):
        invalid_counts.append(record["condition"])

zero_phrase_conditions = [
    record["condition"]
    for record in merged_conditions
    if len(record.get("symptom_phrases", [])) == 0
]

review_conditions = [
    record["condition"]
    for record in merged_conditions
    if bool(record.get("parser_needs_review"))
]

manual_override_records = [
    record["condition"]
    for record in merged_conditions
    if record.get("parser_method") == "manual_override"
]

print("MERGE VALIDATION")
print("----------------")
print("Total records:", len(merged_conditions))
print("With symptom phrases:", len(merged_conditions) - len(zero_phrase_conditions))
print("Zero symptom phrases:", len(zero_phrase_conditions))
print("Still marked for review:", len(review_conditions))
print("Manual override records:", len(manual_override_records))
print("Invalid phrase counts:", len(invalid_counts))

assert len(merged_conditions) == len(parsed_conditions)
assert not invalid_counts

if APPLY_ONLY_WHEN_ZERO:
    assert len(applied) <= len(manual_overrides)

print("\nValidation passed ✅")


MERGE VALIDATION
----------------
Total records: 406
With symptom phrases: 390
Zero symptom phrases: 16
Still marked for review: 32
Manual override records: 60
Invalid phrase counts: 0

Validation passed ✅


## 7. Preview applied manual overrides

In [43]:
PREVIEW_COUNT = 10

shown = 0

for record in merged_conditions:
    if record.get("parser_method") != "manual_override":
        continue

    print("=" * 90)
    print("Condition:", record["condition"])
    print("Symptom phrases:")
    for phrase in record["symptom_phrases"]:
        print("-", phrase)
    print()

    shown += 1
    if shown >= PREVIEW_COUNT:
        break

if shown == 0:
    print("No manual overrides were applied.")


Condition: Acid and chemical burns
Symptom phrases:
- visible scarring
- sight loss
- restricted use of limbs or muscles

Condition: Blisters
Symptom phrases:
- small pockets of fluid that usually form in the upper layers of skin
- filled with a clear fluid (serum)
- filled with blood (blood blisters)
- filled with pus if they become inflamed or infected

Condition: Bow legs and knock knees in children and young people
Symptom phrases:
- a gap between the knees when the ankles are pressed together
- a gap between the ankle bones when the knees are pressed together

Condition: Breast pain
Symptom phrases:
- breast pain
- pain running towards the nipple

Condition: Breast swelling in men
Symptom phrases:
- breast swelling
- the area is often tender

Condition: Cardiac arrest
Symptom phrases:
- loss of consciousness
- stop breathing completely
- stop breathing normally

Condition: Chronic pain
Symptom phrases:
- pain that lasts longer than 12 weeks
- pain beyond the natural healing time



## 8. Save the combined symptom file

In [44]:
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        merged_conditions,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Saved merged symptom dataset:")
print(OUTPUT_PATH)
print("Records saved:", len(merged_conditions))
print("Manual overrides applied:", len(applied))


Saved merged symptom dataset:
C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\preprocessing\conditions_symptom_parsed_with_manual_overrides.json
Records saved: 406
Manual overrides applied: 60


## Result

The generated file is:

```text
app/dataset/output/preprocessing/
conditions_symptom_parsed_with_manual_overrides.json
```

That file is the input to:

```text
3_prepare_model_data.ipynb
```
